# AI Chief of Staff — GRPO Training Notebook

This notebook trains Qwen2.5-3B-Instruct using GRPO (Group Relative Policy Optimisation)
against the AI Chief of Staff RL environment.

**Runtime:** Google Colab T4 (free tier)  
**Model:** Qwen2.5-3B-Instruct (4-bit quantised via Unsloth)  
**Framework:** HuggingFace TRL + Unsloth  

## Episode Structure
Each training episode runs 3 phases in sequence:
1. Email triage (5–15 emails)
2. Calendar conflict resolution (2–5 conflicts)
3. Task delegation (2–5 tasks)

Combined reward = 0.40 × email + 0.35 × calendar + 0.25 × delegation

In [ ]:
# Step 1 — Install dependencies
!pip install unsloth trl transformers accelerate pydantic fastapi uvicorn requests -q

In [ ]:
# Step 2 — Clone the environment
!git clone https://huggingface.co/spaces/YOUR_USERNAME/ai-chief-of-staff /content/cos-env
import sys
sys.path.insert(0, '/content/cos-env')

In [ ]:
# Step 3 — Start the environment server
import subprocess, time
server = subprocess.Popen(
    ['python3', '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '7860'],
    cwd='/content/cos-env'
)
time.sleep(3)
print('Server started')

In [ ]:
# Step 4 — Load model with Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'v_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
)
print('Model loaded')

In [ ]:
# Step 5 — Define reward function for GRPO
import requests, json, re

BASE_URL = 'http://localhost:7860'
TASKS = ['easy_cos', 'medium_cos', 'hard_cos']

def run_episode_reward(completions, task_id='easy_cos', **kwargs):
    """GRPO reward function — runs one episode and returns combined reward."""
    rewards = []
    for completion in completions:
        obs = requests.get(f'{BASE_URL}/reset', params={'task_id': task_id}).json()
        total = 0.0
        steps = 0
        while not obs.get('done') and steps < 50:
            observation = obs.get('observation', obs)
            try:
                match = re.search(r'\{.*\}', completion, re.DOTALL)
                action = json.loads(match.group()) if match else {}
            except Exception:
                action = {'phase': observation.get('phase', 'email')}
            result = requests.post(f'{BASE_URL}/step', json=action).json()
            total += result.get('reward', 0.0)
            obs = result
            steps += 1
        rewards.append(total / max(steps, 1))
    return rewards

print('Reward function defined')

In [ ]:
# Step 6 — Configure and run GRPO training
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir='/content/cos-grpo-output',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    logging_steps=10,
    save_steps=100,
    report_to='none',
)

# TODO: wire GRPOTrainer with run_episode_reward and a prompt dataset
# trainer = GRPOTrainer(
#     model=model,
#     args=training_args,
#     reward_funcs=[run_episode_reward],
#     train_dataset=prompt_dataset,
# )
# trainer.train()
print('Training config ready — fill in dataset and uncomment trainer to run')

In [ ]:
# Step 7 — Evaluate after training
# Run smoke test against trained model and compare to baseline
# Expected improvement: delegation 0.15 → 0.80+
print('After training, run: python3 test_smoke.py')
print('Then run: python3 plots/generate_chart.py to update the comparison chart')